# T03 – Loudness & timbre: first year vs. 2023
*MIR & ML for Audio · TU Berlin · WiSe 2026/27*

| | |
|---|---|
| **Name** | Hallvard Jensen |
| **Student ID** | 0549158 |
| **Corpus (genre)** | Nu Disco/Disco |

**Submission:** work in this notebook – code, figures and text answers all stay here. Before uploading: **Restart Kernel & Run All**, check that every output is shown, then upload this `.ipynb` file.

**Documentation:** document your work directly in the notebook – comments in your code cells and, where helpful, short markdown cells.

### Sources & AI tools
- **Sources** (literature, data, code):
- **LLM coding assistants** (tool/model – how and what for):

## Tasks
1. How has loudness changed between the Top 100 tracks of the first year of your time series and those of 2023?
   - Compute LUFS, loudness range, sample peak, RMS and crest factor based on the previews
   - Compare the central tendency of these measures between both years for this EDMC genre using suitable significance test(s)
   - Summarise the results of the comparison in one sentence a layperson can understand
2. How has timbre changed between the Top 100 tracks of the first year of your time series and those of 2023?
   - Create meaningful categorical timbre profiles based on the previews (clustering method of your choice)
   - Interpret the timbre profiles you created by giving them suitable names
   - Compare the relative dominance of the timbre profiles between both years in this EDMC genre
   - Summarise the results of the comparison in one sentence a layperson can understand

---
## Setup

Expensive steps (feature extraction, separation, model training): save the results to `OUTPUT_DIR` (e.g. as CSV) and load them on the next run instead of recomputing – *Run All* stays fast. Results of earlier sessions are in `output/T02/`, `output/T03/`, …

In [18]:
# import standard packages you'll need

import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(sys.executable)

# ─ add your own additional imports here:
import librosa
import pyloudnorm as pyln
import tqdm
import scipy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

/home/hallvard/miniforge3/envs/mirml/bin/python


In [7]:
# ── CONFIG ───────────────────────────────────────────────────────────────
CORPUS_DIR = Path("data") / "nu-disco-disco"   # ← name of your corpus folder
# ─────────────────────────────────────────────────────────────────────────
SESSION = "T03"
PREVIEW_DIR = CORPUS_DIR / "previews"
OUTPUT_DIR = Path("output") / SESSION      # results of this session
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
CHART_CSV = next(CORPUS_DIR.glob("*.csv"))
print(CHART_CSV)


data/nu-disco-disco/Nu_Disco_Disco.csv


---
## Task 1
**How has loudness changed between the Top 100 tracks of the first year of your time series and those of 2023?**

### 1.1 · Compute LUFS, loudness range, sample peak, RMS and crest factor based on the previews

In [ ]:
# 1.1
def get_loudness_features(id: int):
    """
    Returns a dictionary of loudness features calculated on the preview
    mp3 file corresponding to the provided ID.

    :param id:  ID of preview file
    """
    path = PREVIEW_DIR / f"{id}.mp3"
    if not path.exists():
        return {}
    y, sr = librosa.load(path, sr=None)

    # Make sure all previews are mono
    assert y.ndim == 1

    # Compute RMS for entire track
    rms = np.sqrt(np.mean(np.pow(y, 2)))
    peak = np.max(np.abs(y))

    # Keep all units consistent (dB)
    rms_db = 20 * np.log10(rms)
    peak_db = 20 * np.log10(peak)
    meter = pyln.Meter(sr)

    return {
        "rms": rms_db,
        "lufs": meter.integrated_loudness(y),
        "lra": meter.loudness_range(y),
        "sample_peak": peak_db,
        "crest_factor": peak_db - rms_db,
    }


df_clusters = pd.read_csv(CHART_CSV)

# Select only top 100 tracks from 2019 and 2023
mask = (df_clusters["rank"] <= 100) & (df_clusters["year"].isin([2019, 2023]))
df100 = df_clusters[mask].copy()

# Calculate loudness features for each track. Make sure to compute only
# for unique track ids to avoid doing repeat calculations
df_ids = df100["beatport_track_id"].drop_duplicates()
loud_feat = [get_loudness_features(id) for id in tqdm.tqdm(df_ids)]
df_loud = pd.DataFrame(loud_feat, index=df_ids.index)

# Assign track ids to enable merge of data frames
df_loud["beatport_track_id"] = df_ids

# Merge loudness features with top 100 tracks on track id and save
df_merge = df100.merge(df_loud, on="beatport_track_id")
df_merge.to_csv(OUTPUT_DIR / "loudness.csv", index=False)

  0%|          | 0/192 [00:00<?, ?it/s]

100%|██████████| 192/192 [04:50<00:00,  1.51s/it]


### 1.2 · Compare the central tendency of these measures between both years for this EDMC genre using suitable significance test(s)

In [ ]:
# 1.2
df_clusters = pd.read_csv(OUTPUT_DIR / "loudness.csv")

# Remove any duplicate track ids within the same year
df_clusters = df_clusters.drop_duplicates(["beatport_track_id", "year"])

# Iterate over each loudness feature and perform the Welch t-test
cols = ["rms", "lufs", "lra", "sample_peak", "crest_factor"]
rows = []
for col in cols:
    a = df_clusters.loc[df_clusters["year"] == 2019, col].dropna()
    b = df_clusters.loc[df_clusters["year"] == 2023, col].dropna()

    res = scipy.stats.ttest_ind(a, b, equal_var=False)  # Welch t-test
    conf = res.confidence_interval()
    rows.append(
        {
            "feature": col,
            "p_value": res.pvalue,
            "statistic": res.statistic,
            "conf_low": conf.low,
            "conf_high": conf.high,
        }
    )

res = pd.DataFrame(rows)
print("Welch t-test (2019 vs. 2023):")
print(res)

Welch t-test (2019 vs. 2023):
        feature   p_value  statistic  conf_low  conf_high
0           rms  0.020453  -2.339671 -1.086208  -0.092106
1          lufs  0.010399  -2.591009 -1.020436  -0.137929
2           lra  0.478189  -0.710780 -1.078474   0.507418
3   sample_peak  0.825769  -0.220475 -0.658012   0.525807
4  crest_factor  0.045868   2.011321  0.009709   1.036400


### 1.3 · Summarise the results of the comparison in one sentence a layperson can understand

#### Answer 1.3

The top 100 tracks from 2023 are slightly louder and more compressed than top 100 from 2019, but it is barely noticeable and not strong enough to be conclusive.

---
## Task 2
**How has timbre changed between the Top 100 tracks of the first year of your time series and those of 2023?**

### 2.1 · Create meaningful categorical timbre profiles based on the previews (clustering method of your choice)

In [16]:
# 2.1
def get_timbral_features(id: int):
    """
    Returns a dictionary of timbral features calculated on the preview
    mp3 file corresponding to the provided ID.

    :param id:  ID of preview file
    """
    path = PREVIEW_DIR / f"{id}.mp3"
    if not path.exists():
        return {}
    y, sr = librosa.load(path, sr=None)

    # Make sure all previews are mono
    assert y.ndim == 1

    mean_sc = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
    mean_sf = librosa.feature.spectral_flatness(y=y).mean()
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=5).mean(axis=1)

    return {
        "spectral_centroid": mean_sc,
        "spectral_flatness": mean_sf,
    } | {f"mfcc{i}": mfcc[i] for i in range(1, 5)}


# Calculate timbral features for each track
timb_feat = [get_timbral_features(id) for id in tqdm.tqdm(df_ids)]
df_timb = pd.DataFrame(timb_feat, index=df_ids.index)

# Assign track ids to enable merge of data frames
df_timb["beatport_track_id"] = df_ids

# Merge loudness features with top 100 tracks on track id and save
df_timb = df100.merge(df_timb, on="beatport_track_id")
df_timb.to_csv(OUTPUT_DIR / "timbral.csv", index=False)

100%|██████████| 192/192 [08:39<00:00,  2.71s/it]


In [23]:
df_clusters = pd.read_csv(OUTPUT_DIR / "timbral.csv")
feat_cols = [
    "spectral_centroid",
    "spectral_flatness",
    "mfcc1",
    "mfcc2",
    "mfcc3",
    "mfcc4",
]

df_clusters["spectral_flatness"] = np.log10(df_clusters["spectral_flatness"])
tracks = df_clusters.drop_duplicates("beatport_track_id").dropna(subset=feat_cols).copy()

X = StandardScaler().fit_transform(tracks[feat_cols])
Z = PCA(n_components=0.95, whiten=True, random_state=0).fit_transform(X)

for k in range(2, 9):
    gmm = GaussianMixture(k, n_init=10, random_state=0).fit(Z)
    print(f"k: {k}, bic: {gmm.bic(Z)}")

# Previous runs showed that k=2 has lowest BIC, thus becomes our selection.
K = 2

gmm = GaussianMixture(K, n_init=10, random_state=0).fit(Z)
tracks["profile"] = gmm.predict(Z)

z = pd.DataFrame(X, columns=feat_cols, index=tracks.index)
z["profile"] = tracks["profile"]
print(z.groupby("profile").mean().round(2))

print("Number of tracks per profile:")
print(tracks["profile"].value_counts())

df_clusters = df_clusters.merge(tracks[["beatport_track_id", "profile"]], on="beatport_track_id")
df_clusters.to_csv(OUTPUT_DIR / "timbral_clusters.csv", index=False)

k: 2, bic: 2440.1613189326595
k: 3, bic: 2493.433692428317
k: 4, bic: 2557.784505657557
k: 5, bic: 2561.6393099743254
k: 6, bic: 2654.551396771481
k: 7, bic: 2766.408910741863
k: 8, bic: 2771.3261756240136
         spectral_centroid  spectral_flatness  mfcc1  mfcc2  mfcc3  mfcc4
profile                                                                  
0                     0.05              -0.37  -0.02  -0.00  -0.09  -0.05
1                    -0.17               1.18   0.06   0.01   0.28   0.17
Number of tracks per profile:
profile
0    127
1     40
Name: count, dtype: int64


### 2.2 · Interpret the timbre profiles you created by giving them suitable names

#### Answer 2.2

We chose 2 profiles in 2.1 since this selection had the best BIC value (lowest). Both profiles are very close to spectral centroid average. Thus, there is little difference in brightness. The only separating factor is spectral flatness, where profile 1 is above average and profile 0 is under average. This makes profile 0 less noisy and more tonal, while profile 1 is more noisy:

Profile 0: Tonal (127 tracks)
Profile 1: Noisy (40 tracks)

### 2.3 · Compare the relative dominance of the timbre profiles between both years in this EDMC genre

In [ ]:
# 2.3


### 2.4 · Summarise the results of the comparison in one sentence a layperson can understand

#### Answer 2.4
*Your answer*

---
## Before you upload
- [ ] Name, student ID and genre filled in
- [ ] **Restart Kernel & Run All** runs without errors, all outputs visible
- [ ] Figures labelled (axes, units, title); audio examples embedded where asked